# RAG Evaluation · 02 — RAGAS 框架快速上手

**本 Notebook 目标：** 用 RAGAS (≥ 0.2 API) 框架，5 行核心代码自动化评估 RAG 系统四大指标。

## 学习目标

- ✅ 了解 RAGAS 框架的核心概念与数据格式
- ✅ 配置 DeepSeek 作为 RAGAS 的 LLM 裁判
- ✅ 构建评估数据集并运行四大指标评估
- ✅ 读懂 RAGAS 输出，定位 RAG pipeline 瓶颈
- ✅ 了解评估数据集构建、框架对比、生产环境持续评估


pip install -U ragas langchain-openai "langchain-community==0.3.31"

---
# Part 1 · RAGAS 框架介绍

**RAGAS**（Retrieval Augmented Generation Assessment）是目前最流行的 RAG 评估框架：

- **开源**：GitHub 8k+ stars，持续活跃维护
- **无需人工标注答案**：只需要问题 + 参考上下文，不需要「标准答案」
- **四大指标开箱即用**：Context Recall、Context Precision、Answer Relevancy、Faithfulness
- **支持多种 LLM**：OpenAI、Anthropic、本地模型（通过 LangChain）

| 场景 | RAGAS 的优势 |
|------|-------------|
| 快速验证 | 5 行代码跑完评估 |
| 没有 Ground Truth | 支持无监督评估模式 |
| CI/CD 集成 | Python 原生，易于自动化 |

---

### 本 Notebook 评估的是什么？

**不是在跑真实的 RAG pipeline**，而是用 `EVAL_SAMPLES` 里写死的 4 条样本，模拟「RAG 已跑完」后的结果，再交给 RAGAS 打分。

**各组件角色：**

```mermaid
flowchart LR
    A[EVAL_SAMPLES 静态样本] --> B[RAGAS evaluate]
    C[DeepSeek Chat] -->|LLM 裁判| B
    D[all-MiniLM-L6-v2] -->|AnswerRelevancy 用| B
    B --> E[四大指标分数]
```

| 组件 | 角色 |
|------|------|
| `EVAL_SAMPLES` | 被评估对象（手工构造的 question / contexts / response） |
| DeepSeek Chat | RAGAS 的 LLM 裁判（Faithfulness、Context Recall/Precision 等） |
| all-MiniLM-L6-v2 | 本地 embedding，仅 AnswerRelevancy 算语义相似度 |

---
# Part 2 · 代码实战

### 环境准备

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv


DEEPSEEK_API_KEY = "sk-your-api-key-here"

### 安装 & 配置 RAGAS

In [2]:
import importlib
import sys
import langchain_community
import ragas
from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithoutReference,
    AnswerRelevancy,
    Faithfulness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

DEEPSEEK_BASE_URL = "https://api.deepseek.com"

# LLM 裁判（DeepSeek Chat）
deepseek_chat = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key=DEEPSEEK_API_KEY,
    openai_api_base=DEEPSEEK_BASE_URL,
    temperature=0.0,
    max_retries=2,
)
ragas_llm = LangchainLLMWrapper(deepseek_chat)

# AnswerRelevancy 需要 embedding；使用本地 MiniLM（与 01_vector_limitations 一致，无需 API Key）
local_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
ragas_embeddings = LangchainEmbeddingsWrapper(local_embeddings)

print("✅ RAGAS LLM + Embeddings configured")

C:\Users\86137\AppData\Local\Temp\ipykernel_22308\3115890271.py:6: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\86137\AppData\Local\Temp\ipykernel_22308\3115890271.py:6: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import (
C:\Users\86137\AppData\Local\Temp\ipykernel_22308\3115890271.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\U

✅ RAGAS LLM + Embeddings configured


C:\Users\86137\AppData\Local\Temp\ipykernel_22308\3115890271.py:31: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(local_embeddings)


### 构建评估数据集（4 条样本）

## RAGAS 数据格式

每条样本包含：

| 字段 | 说明 | 哪些指标需要 |
|------|------|-------------|
| `user_input` | 用户问题 | 全部 |
| `retrieved_contexts` | 检索到的文档片段列表 | 全部 |
| `response` | RAG 生成的答案 | Relevancy、Faithfulness |
| `reference` | 标准参考答案 | Context Recall |


In [3]:
EVAL_SAMPLES = [
    {
        "user_input": "什么是 RAG，它的主要作用是什么？",
        "retrieved_contexts": [
            "检索增强生成（RAG）是将信息检索与大语言模型结合的技术。"
            "RAG 先从知识库检索相关文档，再将其作为上下文提供给 LLM 生成答案，从而减少幻觉并提升答案准确性。",
            "大语言模型的幻觉问题是指模型生成听起来合理但实际不准确的内容。"
            "RAG 技术通过提供真实文档作为依据，有效缓解了幻觉问题。",
        ],
        "response": "RAG（检索增强生成）是一种结合信息检索与大语言模型的技术。它的主要作用是通过检索真实文档来减少 LLM 生成虚假信息（幻觉），提升答案的准确性。",
        "reference": "RAG 是检索增强生成技术，将检索与 LLM 结合，主要作用是减少幻觉、提升答案准确性。",
    },
    {
        "user_input": "向量数据库有哪些常见产品？",
        "retrieved_contexts": [
            "向量数据库用于存储和检索高维向量嵌入。"
            "常见产品包括 Pinecone、Weaviate、Milvus 和 Chroma。它们支持近似最近邻（ANN）搜索，适合语义相似度检索。",
        ],
        "response": "常见的向量数据库产品有 Pinecone、Weaviate、Milvus 和 Chroma，均支持近似最近邻搜索，适用于语义检索场景。",
        "reference": "常见向量数据库包括 Pinecone、Weaviate、Milvus、Chroma，支持 ANN 搜索。",
    },
    {
        "user_input": "Transformer 架构是什么时候提出的？有什么创新？",
        "retrieved_contexts": [
            "Transformer 架构由 Vaswani 等人在 2017 年的论文《Attention Is All You Need》中提出。"
            "它完全基于注意力机制，摒弃了循环和卷积结构，在机器翻译任务上取得了突破性成果。",
            "提示工程是设计和优化输入提示以引导 LLM 产生期望输出的技术。常见技巧包括少样本提示和思维链提示。",
        ],
        "response": "Transformer 架构于 2017 年由 Vaswani 等人在论文《Attention Is All You Need》中提出。其核心创新是完全基于注意力机制，去掉了传统的循环（RNN）和卷积（CNN）结构。",
        "reference": "Transformer 于 2017 年提出，核心创新是纯注意力机制，放弃了 RNN 和 CNN 结构。",
    },
    {
        "user_input": "嵌入模型的作用是什么？有哪些常用模型？",
        "retrieved_contexts": [
            "嵌入模型将文本映射到稠密向量空间，使语义相近的文本在向量空间中距离较近。"
            "常用的嵌入模型包括 OpenAI text-embedding-ada-002、BGE 系列和 E5 系列模型。",
        ],
        "response": "嵌入模型的作用是将文本转换为向量，使语义相似的文本在向量空间中更接近，便于进行语义检索。常用模型有 OpenAI 的 text-embedding-ada-002 以及开源的 BGE、E5 系列。",
        "reference": "嵌入模型将文本映射为向量，用于语义检索，代表性模型有 ada-002、BGE、E5 系列。",
    },
]

dataset = EvaluationDataset.from_list(EVAL_SAMPLES)
print(f'✅ 评估数据集构建完成，共 {len(EVAL_SAMPLES)} 条样本')

✅ 评估数据集构建完成，共 4 条样本


### 配置指标 & 运行评估

In [4]:
# 为每个需要 LLM 的指标注入 DeepSeek 裁判
metrics = [
    LLMContextRecall(llm=ragas_llm),                       # 上下文召回率（需 reference）
    LLMContextPrecisionWithoutReference(llm=ragas_llm),    # 上下文精确率（无需 reference）
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),  # 答案相关性（需 embedding）
    Faithfulness(llm=ragas_llm),                           # 忠实度（LLM 裁判）
]

print('正在运行 RAGAS 评估，请稍候...')
results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print('✅ 评估完成')

正在运行 RAGAS 评估，请稍候...


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


✅ 评估完成


### 查看评估结果

In [5]:
import pandas as pd

scores_df = results.to_pandas()

# 整体平均分
print('=' * 65)
print('  RAGAS 评估结果汇总')
print('=' * 65)

metric_cols = [c for c in scores_df.columns
               if c not in ('user_input', 'retrieved_contexts', 'response', 'reference')]

metric_labels = {
    'context_recall': '上下文召回率',
    'context_precision': '上下文精确率',
    'answer_relevancy': '答案相关性',
    'faithfulness': '忠实度',
    'llm_context_recall': '上下文召回率',
    'llm_context_precision_without_reference': '上下文精确率（无参考）',
}

for col in metric_cols:
    mean_val = scores_df[col].mean()
    label = metric_labels.get(col, col)
    flag = '✅' if mean_val >= 0.75 else '⚠️'
    print(f'  {flag} {label} ({col[:30]}): {mean_val:.3f}')

print()
print('  逐样本得分:')
display_cols = ['user_input'] + metric_cols
display(scores_df[display_cols].style.format(
    {c: '{:.3f}' for c in metric_cols}
).background_gradient(subset=metric_cols, cmap='RdYlGn', vmin=0, vmax=1))

  RAGAS 评估结果汇总
  ✅ 上下文召回率 (context_recall): 1.000
  ✅ 上下文精确率（无参考） (llm_context_precision_without_): 1.000
  ⚠️ 答案相关性 (answer_relevancy): 0.507
  ✅ 忠实度 (faithfulness): 0.875

  逐样本得分:


,user_input,context_recall,llm_context_precision_without_reference,answer_relevancy,faithfulness
0,什么是 RAG，它的主要作用是什么？,1.000,1.000,0.292,0.667
1,向量数据库有哪些常见产品？,1.000,1.000,0.918,1.000
2,Transformer 架构是什么时候提出的？有什么创新？,1.000,1.000,0.919,1.000
3,嵌入模型的作用是什么？有哪些常用模型？,1.000,1.000,-0.099,0.833


---
## RAGAS 输出解读

```
   context_recall  context_precision  answer_relevancy  faithfulness
0        0.833333           0.666667          0.921345      0.750000
1        1.000000           0.750000          0.876543      1.000000
2        0.666667           0.500000          0.812341      0.666667
...

{'context_recall': 0.833, 'context_precision': 0.639,
 'answer_relevancy': 0.870, 'faithfulness': 0.805}
```

| 列 | 含义 | 这里的问题 |
|----|------|----------|
| `context_recall` | 每条样本的 Recall | 第 3 条 0.67，检索漏了内容 |
| `context_precision` | 每条样本的 Precision | 第 3 条 0.50，噪声太多 |
| `answer_relevancy` | 答案切题程度 | 整体 0.87，还不错 |
| `faithfulness` | 答案可信度 | 第 1/3 条有幻觉风险 |

---
## 实际 Demo：两个 RAG 版本对比

| 指标 | RAG v1（基础版） | RAG v2（优化版） | 提升 |
|------|---------------|---------------|------|
| Context Recall | 0.64 | **0.83** | +29.7% |
| Context Precision | 0.52 | **0.76** | +46.2% |
| Answer Relevancy | 0.79 | **0.88** | +11.4% |
| Faithfulness | 0.61 | **0.84** | +37.7% |

**v1 → v2 做了什么：**
- chunk_size: 1000 → 512，chunk_overlap: 0 → 128
- top_k: 3 → 6，加了 BGE Reranker 二次排序
- System prompt 增加了"只使用提供的上下文回答"约束

---
# Part 3 · 指标诊断与优化

---
## 指标低了怎么办 — 诊断表

| 指标低 | 阈值参考 | 可能原因 | 优化方向 |
|--------|---------|---------|---------|
| **Context Recall** | < 0.75 | top_k 太小；chunk 太大；embedding 召回差 | 加大 top_k；缩小 chunk_size；换更强 embedding |
| **Context Precision** | < 0.65 | 检索结果含大量无关片段 | 加 Reranker；提高相似度阈值；改善文档质量 |
| **Answer Relevancy** | < 0.80 | Prompt 没有引导模型直接回答；答案太长绕弯 | Query 重写；优化 response prompt |
| **Faithfulness** | < 0.75 | System prompt 约束弱；检索内容不足 | 强化 system prompt；先提高 Recall |

> **优先级建议：先修 Recall → 再修 Faithfulness → 再修 Precision → 最后 Relevancy**

---
## Context Recall 低 — 怎么修

**根本原因：检索器没找到回答问题所需的关键信息**

```python
# 1. 增大 top_k
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 从 3 改到 6

# 2. 缩小 chunk_size，增加 overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,      # 从 1000 缩小
    chunk_overlap=128,   # 从 0 增加
)

# 3. 混合检索：向量 + BM25 关键词
from langchain.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)

# 4. 换更强的 Embedding 模型
# text-embedding-ada-002 → text-embedding-3-large
# 或中文专用：BAAI/bge-large-zh-v1.5
```

---
## Context Precision 低 — 怎么修

**根本原因：检索结果混入了太多与问题无关的噪声片段**

```python
# 方案 1：加 Reranker 二次排序
from langchain.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-multilingual-v3.0", top_n=3)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

# 方案 2：提高相似度阈值，过滤低质量结果
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.75, "k": 6}
)
```

---
## Faithfulness 低 — 怎么修

**根本原因：LLM 在生成答案时脱离了检索内容，加入了自己的「发挥」**

```python
# 方案 1：强化 System Prompt 约束
"""你是一个严格基于文档的问答助手。

规则：
1. 只使用【参考文档】中的信息来回答问题
2. 如果文档中没有相关信息，回答"根据现有文档，无法回答此问题"
3. 不要添加文档中没有的任何信息
4. 引用原文时保持准确，不要改写核心事实

【参考文档】
{context}

用户问题：{question}"""

# 方案 2：先提高 Recall（LLM 有足够原材料就不需要「补充」）
# 方案 3：降低 LLM temperature
llm = ChatOpenAI(model="gpt-4o", temperature=0)
```

---
## Answer Relevancy 低 — 怎么修

**根本原因：答案没有直接回答用户的问题（答非所问或绕弯太多）**

```python
# 方案 1：Query 重写 — 让问题更精确
"""将以下用户问题改写为更清晰、更具体的检索查询：
原始问题：{question}
改写后的查询（直接给出，不要解释）："""

# 方案 2：HyDE（假设文档嵌入）— 先生成假设答案再检索
"""请为以下问题写一个假设性的简短答案（50字以内）：
问题：{question}"""
# 用假设答案的 embedding 去检索，比问题 embedding 更接近答案空间

# 方案 3：优化 Response Prompt，要求直接回答
"""请直接回答用户的问题。
第一句话必须直接给出答案，不要铺垫。
问题：{question}
上下文：{context}"""
```

---
# Part 4 · 进阶：生产环境评估

---
## 常见错误 Top 5

**No.1 没有 Ground Truth 就跑评估**
> Recall 和 Precision 必须有标注的参考上下文，否则结果没有意义。至少准备 50 条标注数据。

**No.2 用同一个 LLM 既生成答案又评估**
> 自己评自己的分必然虚高。评估 LLM 应比生成 LLM 更强，或使用独立的评估模型。

**No.3 只看 Faithfulness，忽略 Recall**
> Faithfulness 高但 Recall 低 = 答案很诚实但信息不全。用户会觉得「答得不够」。

**No.4 只看平均分，不看样本分布**
> 平均 0.80 可能藏着 30% 的样本 Faithfulness = 0.0（完全幻觉）。务必看 min、p10 分位数。

**No.5 评估集太小（< 20 条）**
> 样本不足时指标波动极大，换几条样本分数可能差 0.2+。生产系统最少 100 条。

---
## 评估数据集构建 — 自动生成 50 条 Q&A

```python
from ragas.testset import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context

generator = TestsetGenerator.from_langchain(
    generator_llm=ChatOpenAI(model="gpt-4o"),
    critic_llm=ChatOpenAI(model="gpt-4o"),
    embeddings=OpenAIEmbeddings()
)

testset = generator.generate_with_langchain_docs(
    documents=docs,
    test_size=50,
    distributions={
        simple: 0.5,         # 50% 简单问题
        reasoning: 0.25,     # 25% 推理问题
        multi_context: 0.25  # 25% 需要多文档的问题
    }
)
```

- **方法 1**：RAGAS 自动生成（推荐快速启动）
- **方法 2**：领域专家手工标注（质量最高，成本最高）
- **方法 3**：从真实用户日志中抽取问题，人工补充 reference_contexts

---
## RAGAS vs 其他框架

| 维度 | **RAGAS** | **TruLens** | **DeepEval** |
|------|----------|------------|-------------|
| **定位** | RAG 专用评估 | LLM App 全链路追踪 | 通用 LLM 测试框架 |
| **安装** | `pip install ragas` | `pip install trulens` | `pip install deepeval` |
| **四大 RAG 指标** | ✅ 原生支持 | 部分支持 | ✅ 支持 |
| **无监督评估** | ✅ 支持 | ❌ 需要 GT | ✅ 支持 |
| **UI Dashboard** | ❌ 无（用 DataFrame） | ✅ 内置 UI | ✅ 内置 UI |
| **CI 集成** | ✅ 简单 | 中等 | ✅ 简单 |
| **社区活跃度** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |

> **推荐策略：** 快速评估用 RAGAS，需要实验追踪和 UI 用 TruLens，需要更多指标类型用 DeepEval。

---
## 生产环境持续评估

```python
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your-key"
os.environ["LANGCHAIN_PROJECT"] = "rag-evaluation"

def run_weekly_evaluation(rag_pipeline, eval_dataset):
    """每周跑一次，检测 RAG 质量是否退化"""
    results = evaluate(dataset=eval_dataset, metrics=metrics)
    scores = results.to_pandas().mean()

    thresholds = {
        "context_recall": 0.80,
        "faithfulness": 0.80,
        "answer_relevancy": 0.85,
    }
    for metric, threshold in thresholds.items():
        if scores[metric] < threshold:
            alert(f"[RAG 质量告警] {metric} = {scores[metric]:.3f} < {threshold}")
```

---
# 关键 Takeaways

1. **量化是工程的基础** — 没有指标，RAG 优化全是感觉，无法说服团队和老板继续投入

2. **四个指标覆盖两个维度** — 检索端（Recall + Precision）和生成端（Relevancy + Faithfulness），缺一不可

3. **优化顺序有讲究** — 先 Recall（没材料什么都做不了）→ 再 Faithfulness（防幻觉）→ 再 Precision → 最后 Relevancy

4. **RAGAS 让评估变简单** — 5 行代码跑完四个指标，无需大量人工标注，可接入 CI/CD

5. **评估数据集质量决定评估质量** — 最少 50 条，覆盖不同难度和类型，定期更新

---
## 课程小结

| 指标 | 核心问题 | 公式 | 低分怎么修 |
|------|---------|------|----------|
| **Context Recall** | 该找的都找到了吗 | \|GT∩Ret\|/\|GT\| | 加 top_k、缩 chunk、换 embedding |
| **Context Precision** | 找的都相关吗 | \|GT∩Ret\|/\|Ret\| | 加 Reranker、提相似度阈值 |
| **Answer Relevancy** | 答的切题吗 | 反向问题相似度均值 | Query 重写、优化 prompt |
| **Faithfulness** | 有没有瞎编 | 被支持的 claim 比例 | 强化 system prompt、先提 Recall |

> **一句话总结：这四个数字是你优化 RAG 的指南针——没有它们，你在黑暗中摸索。**